# Exercise — tune a model end to end

Work through this on your own. A worked solution is in `solutions/`.

You will tune a `HistGradientBoostingClassifier` on the same penguin data. It is a different
model from the one used in the teaching notebooks, deliberately — the point is that the
procedure does not change.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])

X = penguins.drop(columns="sex")
y = penguins["sex"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RSEED, stratify=y
)

numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy="most_frequent"),
                            OneHotEncoder(handle_unknown="ignore"))

preprocessor = make_column_transformer(
    (numeric, make_column_selector(dtype_include="number")),
    (categorical, make_column_selector(dtype_exclude="number")),
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

print(f"{len(X_train)} training penguins, {len(X_test)} held out")

249 training penguins, 84 held out


## Task 1 — find the tunable names

Build a pipeline with `preprocessor` and a `HistGradientBoostingClassifier`, then print the
parameter names you could tune on the classifier step.

*Hint: `get_params()`, and filter for names starting with `classifier__`.*

In [3]:
hgb = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=RSEED)),
])

for name, value in sorted(hgb.get_params().items()):
    if name.startswith("classifier__"):
        print(f"  {name:42s} = {value}")

  classifier__categorical_features           = from_dtype
  classifier__class_weight                   = None
  classifier__early_stopping                 = auto
  classifier__interaction_cst                = None
  classifier__l2_regularization              = 0.0
  classifier__learning_rate                  = 0.1
  classifier__loss                           = log_loss
  classifier__max_bins                       = 255
  classifier__max_depth                      = None
  classifier__max_features                   = 1.0
  classifier__max_iter                       = 100
  classifier__max_leaf_nodes                 = 31
  classifier__min_samples_leaf               = 20
  classifier__monotonic_cst                  = None
  classifier__n_iter_no_change               = 10
  classifier__random_state                   = 42
  classifier__scoring                        = loss
  classifier__tol                            = 1e-07
  classifier__validation_fraction            = 0.1
  classifier__v

## Task 2 — a baseline

Before tuning anything, cross-validate the model with its **default** settings.

Write down the number. If your tuned model does not beat it, the tuning was not worth doing.

In [4]:
baseline_scores = cross_val_score(hgb, X_train, y_train, cv=5, scoring="accuracy")
baseline = baseline_scores.mean()
print(f"default HistGradientBoosting: {baseline:.4f} ± {baseline_scores.std():.4f}")

default HistGradientBoosting: 0.9116 ± 0.0450


## Task 3 — grid search

Tune `learning_rate` and `max_leaf_nodes` with `GridSearchCV`.

Before you run it, **work out how many fits your grid will cost** and print that number.

Report `best_params_`, `best_score_`, and how much you improved on the baseline.

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__learning_rate": [0.01, 0.05, 0.1, 0.5],
    "classifier__max_leaf_nodes": [5, 15, 31, 63],
}

combinations = 4 * 4
print(f"{combinations} combinations × 5 folds = {combinations * 5} fits\n")

grid = GridSearchCV(hgb, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"best params : {grid.best_params_}")
print(f"best cv     : {grid.best_score_:.4f}")
print(f"baseline    : {baseline:.4f}")
print(f"improvement : {grid.best_score_ - baseline:+.4f}")

16 combinations × 5 folds = 80 fits

best params : {'classifier__learning_rate': 0.5, 'classifier__max_leaf_nodes': 15}
best cv     : 0.9277
baseline    : 0.9116
improvement : +0.0161


## Task 4 — randomized search

Do the same job with `RandomizedSearchCV` and a budget of 30, sampling `learning_rate` from
`loguniform` and `max_leaf_nodes` from `randint`.

Did it find a better combination than the grid? Did it cost more or less?

In [6]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__learning_rate": loguniform(1e-2, 1e0),   # 0.01 to 1
    "classifier__max_leaf_nodes": randint(5, 64),      # 5 to 63
}

rand = RandomizedSearchCV(hgb, param_distributions, n_iter=30, cv=5,
                          scoring="accuracy", random_state=RSEED, n_jobs=-1)
rand.fit(X_train, y_train)

print(f"grid   : {grid.best_score_:.4f} from {16 * 5} fits")
print(f"random : {rand.best_score_:.4f} from {30 * 5} fits")
print(f"\nrandom best params: {rand.best_params_}")

grid   : 0.9277 from 80 fits
random : 0.9316 from 150 fits

random best params: {'classifier__learning_rate': np.float64(0.029210748185657155), 'classifier__max_leaf_nodes': 51}


## Task 5 — how confident are you?

Run nested cross-validation on your grid search and report the estimate with its spread.

Then answer, in a markdown cell: **would you be comfortable claiming this model beats the
baseline?** Use the spread to justify your answer, not just the means.

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_validate

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSEED)

tuner = GridSearchCV(hgb, param_grid, cv=inner_cv, scoring="accuracy", n_jobs=-1)
nested = cross_validate(tuner, X_train, y_train, cv=outer_cv, scoring="accuracy")

baseline_nested = cross_validate(hgb, X_train, y_train, cv=outer_cv, scoring="accuracy")

print(f"tuned    : {nested['test_score'].mean():.4f} ± {nested['test_score'].std():.4f}")
print(f"baseline : {baseline_nested['test_score'].mean():.4f} ± {baseline_nested['test_score'].std():.4f}")

tuned    : 0.9038 ± 0.0341
baseline : 0.9158 ± 0.0318


### Would I claim the tuned model beats the baseline?

**No.** Under nested cross-validation the tuned model scores 0.904 ± 0.034,
while the untuned default scores 0.916 ± 0.032. The intervals overlap heavily,
so tuning made no real difference — the tuned model is even slightly lower.

The grid search reported `best_score_ = 0.928`, an apparent +0.016 over the
baseline. That number was the maximum of 16 noisy cross-validation scores, so it
was biased upward by selection bias, not a genuine gain.

**Next step:** the defaults are already close to what this model can do on this
data, so my next hour is better spent on new features or a different model than
on a bigger grid.

## Stretch

Add the preprocessing choice to the search space — try both `"mean"` and `"median"`
imputation as part of the same grid. Does the best imputation strategy depend on which
classifier settings are used?

In [8]:
stretch_grid = {
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_leaf_nodes": [15, 31],
    "preprocessor__pipeline-1__simpleimputer__strategy": ["mean", "median"],
}

stretch = GridSearchCV(hgb, stretch_grid, cv=5, scoring="accuracy", n_jobs=-1)
stretch.fit(X_train, y_train)

results = pd.DataFrame(stretch.cv_results_)
pivot = results.pivot_table(
    index=["param_classifier__learning_rate", "param_classifier__max_leaf_nodes"],
    columns="param_preprocessor__pipeline-1__simpleimputer__strategy",
    values="mean_test_score",
)
pivot.round(4)

param_preprocessor__pipeline-1__simpleimputer__strategy             mean  \
param_classifier__learning_rate param_classifier__max_leaf_nodes           
0.05                            15                                0.9275   
                                31                                0.9275   
0.10                            15                                0.9115   
                                31                                0.9115   

param_preprocessor__pipeline-1__simpleimputer__strategy           median  
param_classifier__learning_rate param_classifier__max_leaf_nodes          
0.05                            15                                0.9195  
                                31                                0.9195  
0.10                            15                                0.9116  
                                31                                0.9116